In [1]:
import pandas as pd
import numpy as np

from surprise import Reader
from surprise import Dataset
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [2]:
data = pd.read_csv("ratings.csv")

data.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
data.shape

(100836, 4)

In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [5]:
print("Users :", data["userId"].nunique())
print("Movies :", data["movieId"].nunique())
print("Ratings :", len(data))

Users : 610
Movies : 9724
Ratings : 100836


In [6]:
data.isnull().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

In [7]:
reader = Reader(rating_scale=(0.5,5))

dataset = Dataset.load_from_df(
    data[["userId","movieId","rating"]],
    reader
)

In [8]:
trainset, testset = train_test_split(
    dataset,
    test_size=0.2,
    random_state=10
)

In [9]:
algo = SVD()

algo.fit(trainset)

In [10]:
predictions = algo.test(testset)

accuracy.rmse(predictions)

RMSE: 0.8709


np.float64(0.8708811424500429)

In [11]:
uid = 1
iid = 50

pred = algo.predict(uid, iid)

print(pred)

user: 1          item: 50         r_ui = None   est = 5.00   {'was_impossible': False}


In [12]:
movies = data["movieId"].unique()

result = []

for i in movies:
    p = algo.predict(1, i)
    result.append([i, p.est])

In [13]:
result = sorted(result, key=lambda x:x[1], reverse=True)

result[:10]

[[np.int64(50), 5],
 [np.int64(2329), 5],
 [np.int64(2571), 5],
 [np.int64(318), 5],
 [np.int64(58559), 5],
 [np.int64(1272), 5],
 [np.int64(904), 5],
 [np.int64(1225), 5],
 [np.int64(1259), 5],
 [np.int64(4226), 5]]

In [14]:
top = pd.DataFrame(result[:10])

top.columns = ["MovieId","Predicted Rating"]

top

,MovieId,Predicted Rating
0,50,5
1,2329,5
2,2571,5
3,318,5
4,58559,5
5,1272,5
6,904,5
7,1225,5
8,1259,5
9,4226,5


In [15]:
top.to_csv("recommendation_output.csv", index=False)